# CodeAlpha Task 4: Sales Prediction using Python

**Internship Provider:** CodeAlpha  
**Domain:** Data Science and Machine Learning  
**Project Objective:** Build, evaluate, and interpret machine learning regression models to forecast product sales based on advertising expenditures across Television, Radio, and Newspaper channels.  
**Author:** Vaishnav Sreejan  

---

### Problem Statement
Businesses invest significant capital into advertising campaigns across varied media channels. However, marketing budgets are constrained, and understanding which channels provide the highest sales return is crucial for maximizing Return on Marketing Investment (ROMI). In this project, we employ exploratory data analysis and supervised regression algorithms to quantify channel effectiveness and accurately predict sales volume.


## 1. Import Required Libraries
We load essential libraries for data manipulation (`pandas`, `numpy`), statistical visualization (`matplotlib`, `seaborn`), and machine learning (`scikit-learn`, `joblib`).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# Set visualization styles
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Dataset Loading and Initial Inspection
The dataset contains 200 market campaigns with advertising spend ($1,000s) on **TV**, **Radio**, and **Newspaper**, along with resulting **Sales** (thousands of units).


In [1]:
# Load dataset from data folder
df = pd.read_csv('../data/sales_data.csv')

# Drop unnamed index column if present
unnamed_cols = [c for c in df.columns if 'unnamed' in str(c).lower() or str(c).strip() == '']
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Dataset Shape: 200 rows, 4 columns


## 3. Data Cleaning & Integrity Checks
We check for missing values, duplicate entries, data types, and verify that advertising spends are non-negative.


In [1]:
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Duplicates Check ---")
print(f"Total duplicate records: {df.duplicated().sum()}")

print("\n--- Summary Statistics ---")
df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]


## 4. Exploratory Data Analysis (EDA)
### Sales Distribution and Channel Relationships
Let us examine the target variable distribution and how each advertising medium correlates with sales volume.


In [1]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
channels = [('TV', '#2b5c8f'), ('Radio', '#d95f02'), ('Newspaper', '#1b9e77')]

for i, (channel, color) in enumerate(channels):
    corr_val = df[channel].corr(df['Sales'])
    sns.regplot(data=df, x=channel, y='Sales', ax=axes[i], color=color, scatter_kws={'alpha': 0.65, 's': 40}, line_kws={'linewidth': 2, 'color': '#222222'})
    axes[i].set_title(f"{channel} vs Sales (r = {corr_val:.3f})", fontweight='bold')
    axes[i].set_xlabel(f"{channel} Spend ($1,000s)")
    if i == 0:
        axes[i].set_ylabel("Sales (Thousands of Units)")

fig.suptitle("Advertising Spend vs Product Sales across Channels", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


### Correlation Heatmap & Channel Spend Comparison
Understanding inter-feature collinearity and overall budget share across channels.


In [1]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Correlation Heatmap
sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='coolwarm', square=True, ax=ax1, linewidths=1)
ax1.set_title("Correlation Matrix", fontweight='bold')

# Budget Share Donut Chart
channels = ['TV', 'Radio', 'Newspaper']
spends = [df[c].sum() for c in channels]
ax2.pie(spends, labels=channels, autopct='%1.1f%%', startangle=140, colors=['#2b5c8f', '#d95f02', '#1b9e77'], wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))
ax2.set_title("Overall Marketing Budget Share", fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Train-Test Split (80/20)
We partition the dataset into 80% training data (160 samples) and 20% hold-out test data (40 samples). Using a fixed random seed (`random_state=42`) ensures full reproducibility.


In [1]:
feature_cols = ['TV', 'Radio', 'Newspaper']
X = df[feature_cols]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set:  {X_test.shape[0]} samples")


Training Set: 160 samples
Testing Set:  40 samples


## 6. Model Training and Benchmarking
We train and compare four candidate regression models:
1. **Linear Regression** (Parametric baseline)
2. **Decision Tree Regressor** (Non-linear single tree)
3. **Random Forest Regressor** (Ensemble bagging)
4. **Gradient Boosting Regressor** (Ensemble boosting)

All models are integrated into pipelines with `StandardScaler` to ensure robust standardization without data leakage.


In [1]:
models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())]),
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('reg', DecisionTreeRegressor(max_depth=5, random_state=42))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('reg', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42))]),
    'Gradient Boosting': Pipeline([('scaler', StandardScaler()), ('reg', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))])
}

results = []
fitted_pipelines = {}
test_preds = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    tr_pred = pipe.predict(X_train)
    te_pred = pipe.predict(X_test)
    test_preds[name] = te_pred
    
    results.append({
        'Model': name,
        'Train R2': round(r2_score(y_train, tr_pred), 4),
        'Test R2': round(r2_score(y_test, te_pred), 4),
        'Test RMSE': round(root_mean_squared_error(y_test, te_pred), 4),
        'Test MAE': round(mean_absolute_error(y_test, te_pred), 4),
        'Test MSE': round(mean_squared_error(y_test, te_pred), 4)
    })

results_df = pd.DataFrame(results).sort_values(by='Test R2', ascending=False).reset_index(drop=True)
results_df


## 7. Model Evaluation & Diagnostic Plots
Let us inspect the top model (`Gradient Boosting`) by evaluating Actual vs. Predicted sales and residual errors.


In [1]:
best_model = 'Gradient Boosting'
y_pred_best = test_preds[best_model]
residuals = y_test - y_pred_best

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
ax1.scatter(y_test, y_pred_best, color='#2b5c8f', s=60, edgecolors='w', alpha=0.8)
min_v, max_v = min(y_test.min(), y_pred_best.min()) - 1, max(y_test.max(), y_pred_best.max()) + 1
ax1.plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=2, label='Ideal Fit (y = x)')
ax1.set_title(f"Actual vs Predicted Sales ({best_model})", fontweight='bold')
ax1.set_xlabel('Actual Sales (k units)')
ax1.set_ylabel('Predicted Sales (k units)')
ax1.legend()

# Residual Plot
ax2.scatter(y_pred_best, residuals, color='#7570b3', s=50, edgecolors='w', alpha=0.8)
ax2.axhline(0, color='r', linestyle='--', linewidth=2)
ax2.set_title("Residuals vs Predicted Values", fontweight='bold')
ax2.set_xlabel('Predicted Sales (k units)')
ax2.set_ylabel('Residual Error')

plt.tight_layout()
plt.show()


## 8. Feature Importance Analysis
We examine the relative feature contribution to sales predictions derived from the Gradient Boosting model.


In [1]:
best_pipeline = fitted_pipelines[best_model]
regressor = best_pipeline.named_steps['reg']
importances = regressor.feature_importances_

imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances,
    'Share_Pct': (importances / importances.sum() * 100).round(2)
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print(imp_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=imp_df, x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title("Relative Feature Importance (Gradient Boosting)", fontweight='bold')
plt.xlabel("Importance Score")
plt.show()


## 9. Model Serialization & Verification
We save the fitted pipeline into `models/sales_prediction_model.joblib` and test reloading for inference.


In [1]:
# Save pipeline
model_path = '../models/sales_prediction_model.joblib'
joblib.dump(best_pipeline, model_path)
print(f"Model pipeline saved to: {model_path}")

# Reload and test
loaded_pipe = joblib.load(model_path)
test_campaign = pd.DataFrame([{'TV': 230.1, 'Radio': 37.8, 'Newspaper': 69.2}])
prediction = loaded_pipe.predict(test_campaign)
print(f"Test Campaign Input: {test_campaign.to_dict(orient='records')[0]}")
print(f"Predicted Sales: {prediction[0]:.2f} thousand units")


Model pipeline saved to: ../models/sales_prediction_model.joblib
Test Campaign Input: {'TV': 230.1, 'Radio': 37.8, 'Newspaper': 69.2}
Predicted Sales: 22.05 thousand units


## 10. Key Insights & Business Recommendations

1. **TV Dominates Awareness:** TV accounts for **61.26%** of model predictive importance and has a strong correlation with sales ($r = 0.782$).
2. **Radio as an Effective Multiplier:** Radio accounts for **38.09%** of model importance, acting as a cost-effective frequency builder alongside TV.
3. **Newspaper Spend Optimization:** Print advertising contributed less than **1%** to model predictive weight ($0.65%$) and showed weak direct correlation. Marketing budget should be shifted towards TV and digital/radio.
4. **High Predictive Confidence:** The Gradient Boosting model explains **98.31%** of variance in test sales with an RMSE of **0.73k units**, making it an effective decision-support tool.
